# Windows en Spark

Iniciar Sesión de Spark (Spark Session)

---

In [ ]:
from pyspark.sql import SparkSession

# crear la sesión
spark = SparkSession \
        .builder \
        .appName("DataFrames Basics") \
        .master("local[*]") \
        .getOrCreate()

spark.version

In [ ]:
spark

In [ ]:
# Para optimización de conversión a Pandas
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [ ]:
# Importar funciones sql
from pyspark.sql.functions import *

## Examples

### Particionado con Windows 

In [ ]:
developersDF = spark.read.option("header", "true").option("inferSchema", "true").csv("developers.csv")
paychecksDF = spark.read.option("header", "true").option("inferSchema", "true").csv("paychecks.csv")
leadsDF = spark.read.option("header", "true").option("inferSchema", "true").csv("leads.csv")
rolesDF = spark.read.option("header", "true").option("inferSchema", "true").csv("roles.csv")
gamesDF = spark.read.option("inferSchema", "true").option("header", "true").csv("games.csv")

In [ ]:
rolesDF.show(2)

+------+--------+----------+----------+
|emp_no|   title| from_date|   to_date|
+------+--------+----------+----------+
| 10010|Engineer|1996-11-24|9999-01-01|
| 10020|Engineer|1997-12-30|9999-01-01|
+------+--------+----------+----------+
only showing top 2 rows



Rol Más Reciente por Desarrollador

In [ ]:
# Filtrar para ver el historial de un desarrollador de ejemplo
# Nota: La columna 'to_date' debe estar en formato YYYY-MM-DD para ordenar correctamente.
# Asumimos que "9999-01-01" es la fecha más lejana, es decir, el rol actual.
rolesDF.filter(col("dev_id") == 1004).show(4)

+------+---------------+----------+----------+
|emp_no|          title| from_date|   to_date|
+------+---------------+----------+----------+
| 10040|       Engineer|1993-02-14|1999-02-14|
| 10040|Senior Engineer|1999-02-14|9999-01-01|
+------+---------------+----------+----------+



In [ ]:
from pyspark.sql.functions import col, max, row_number, udf, regexp_replace
from pyspark.sql.window import Window
from pyspark.sql.types import LongType, StringType

In [ ]:
# Definición de la Ventana: Particionar por dev_id y ordenar por to_date (descendente)
byDeveloper = Window.partitionBy("dev_id").orderBy(col("to_date").desc())

mostRecentRoleDF = rolesDF.withColumn("datesOrder", row_number().over(byDeveloper)) \
    .filter(col("datesOrder") == 1) \
    .select("dev_id", "role_name", "to_date")

mostRecentRoleDF.filter(col("dev_id") == 1004).show(4)

+------+---------------+----------+
|emp_no|          title|   to_date|
+------+---------------+----------+
| 10040|Senior Engineer|9999-01-01|
+------+---------------+----------+



In [ ]:
# Para el ejemplo anterior que vimos en los joins, si utilizamos la partición de ventanas, no es necesario realizar el paso anterior (filtrar la fecha máxima y luego realizar el join sobre la misma tabla).

Los 3 Pagos Máximos por Rol Actual

In [ ]:
# 2.1 Unimos los pagos con el rol más reciente (similar a unir salarios y títulos)
# Primero, aseguramos que 'amount' sea LongType (como 'salary' a 'long')
paychecksDF = paychecksDF.withColumn("amount", col("amount").cast(LongType()))

bestPaidPerRoleRawDF = paychecksDF.join(
    mostRecentRoleDF,
    # Unimos por ID de desarrollador y usamos la fecha de finalización del rol como filtro para asegurar que el pago cae en ese rol.
    # Esta es una simplificación del ejemplo original, ya que no tenemos 'from_date' en el DF de pagos.
    paychecksDF.dev_id == mostRecentRoleDF.dev_id,
    "inner"
).drop(mostRecentRoleDF.dev_id).drop("to_date", "pay_date")

print("Pagos unidos a roles recientes:")
bestPaidPerRoleRawDF.show(3)


+------+---------------+
|salary|          title|
+------+---------------+
| 72668|Senior Engineer|
+------+---------------+

+------+---------------+
|salary|          title|
+------+---------------+
| 80324|       Engineer|
| 47017|       Engineer|
| 88806|Senior Engineer|
+------+---------------+
only showing top 3 rows



In [ ]:
# 2.2 Aplicamos Window Partitioning
# Definición de la Ventana: Particionar por role_name y ordenar por amount (descendente)
byRole = Window.partitionBy("role_name").orderBy(col("amount").desc())

bestPaidPerRoleDF = bestPaidPerRoleRawDF.withColumn("rank_paycheck", row_number().over(byRole)).filter(col("rank_paycheck") <= 3)

print("Los 3 mejores pagos por rol actual:")
bestPaidPerRoleDF.show(6)

+------+------------------+-----------+
|salary|             title|rank_salary|
+------+------------------+-----------+
|101622|Assistant Engineer|          1|
| 92674|Assistant Engineer|          2|
| 92034|Assistant Engineer|          3|
|130939|          Engineer|          1|
|121819|          Engineer|          2|
|120417|          Engineer|          3|
+------+------------------+-----------+
only showing top 6 rows



### UDFs

In [ ]:
gamesDF.show(3)

In [ ]:
# Creamos la udf
def is_unreal(engine):
    if "Unreal" in engine:
        return "Yes"
    else:
        return "No"

# Registrar UDF
is_unreal_udf = udf(is_unreal, StringType())

# Usar UDF en el DataFrame
df_with_udf = gamesDF.withColumn("uses_unreal", is_unreal_udf(gamesDF["engine"]))

df_with_udf.show()

## Ejercicios de Particionado de Windows
1. Rol más reciente por desarrollador: Usando el archivo roles.csv, obtén para cada desarrollador (dev_id) su rol más reciente según from_date.

2. Diferencia de duración de roles por desarrollador: Usando roles.csv, calcula la duración de cada rol en días (to_date - from_date). Luego, calcula la diferencia de duración de cada rol respecto al rol de menor duración del mismo desarrollador.

Ejercicio 1

Ejercicio 2

## Ejercicios UDFs
1. Elige uno de los DF que tenemos, define dos UDF propios y aplícalos (con un withColumn) al DF. Muestra los resultados. 